# Entrenamiento — Clasificador de pingüinos

Entrena varios modelos candidatos y los guarda en `/models`, el volumen
compartido con el servicio de API. El API observa esa carpeta con
`watchdog` y recarga automáticamente cualquier modelo nuevo — no hace
falta reiniciar el contenedor del API.

Para probar el hot-reload: vuelve a correr este notebook (o cambia algún
hiperparámetro) y observa `GET /models` en el API — el modelo actualizado
debe aparecer solo, sin reiniciar nada.

In [1]:
import json
import os
import pickle
import tempfile
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("/app/data/penguins.csv")
MODELS_DIR = Path("/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "island", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g", "sex",
]
TARGET_COL = "species"

## Guardado atómico

`pickle.dump` directo al archivo final no es seguro aquí: el API está
observando `/models` con watchdog, y si lee el archivo mientras se está
escribiendo, obtiene un pickle corrupto. La solución estándar es escribir
a un archivo temporal en el mismo directorio y luego renombrarlo — el
rename es atómico a nivel de sistema de archivos, así que el API nunca ve
un archivo a medio escribir.

In [2]:
def atomic_pickle_dump(obj, path: Path):
    fd, tmp_path = tempfile.mkstemp(dir=path.parent, prefix=".tmp_")
    try:
        with os.fdopen(fd, "wb") as f:
            pickle.dump(obj, f)
        os.replace(tmp_path, path)
    except Exception:
        os.unlink(tmp_path)
        raise


def atomic_json_dump(obj, path: Path):
    fd, tmp_path = tempfile.mkstemp(dir=path.parent, prefix=".tmp_")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2)
        os.replace(tmp_path, path)
    except Exception:
        os.unlink(tmp_path)
        raise

## Etapa 1: Procesamiento de datos

In [3]:
# 1.1 Carga de datos
df = pd.read_csv(DATA_PATH, index_col=0)

# 1.2 Limpieza: se eliminan filas con nulos en las columnas usadas
df_clean = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

# 1.3 Codificación de variables categóricas
le_island = LabelEncoder()
le_sex = LabelEncoder()
df_clean["island"] = le_island.fit_transform(df_clean["island"])
df_clean["sex"] = le_sex.fit_transform(df_clean["sex"])

X = df_clean[FEATURE_COLS]
y = df_clean[TARGET_COL]

# 1.4 División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((266, 6), (67, 6))

## Etapa 2: Creación de modelo

Se entrenan varios modelos candidatos (2.1), se evalúan (2.3) y se
guardan todos (2.4) — la experimentación suele arrojar más de un modelo
con buen desempeño, y el API permite elegir cuál usar en producción.

In [4]:
# 2.1 Modelos candidatos. KNN y regresión logística son sensibles a la
# escala de las variables (body_mass_g en miles vs. bill_length_mm en
# decenas), por eso van en un pipeline con StandardScaler.
candidate_models = {
    "decision_tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "logistic_regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
}

metrics = {}

for name, model in candidate_models.items():
    # 2.2 Entrenamiento
    model.fit(X_train, y_train)

    # 2.3 Evaluación
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    metrics[name] = {"accuracy": round(float(acc), 4)}
    print(f"{name}: accuracy={acc:.4f}")

    # 2.4 Guardado atómico del modelo en el volumen compartido
    atomic_pickle_dump(model, MODELS_DIR / f"{name}.pkl")

decision_tree: accuracy=0.9403
random_forest: accuracy=1.0000
knn: accuracy=0.9851


logistic_regression: accuracy=1.0000


In [5]:
# Encoders necesarios para transformar island/sex en el API
atomic_pickle_dump({"island": le_island, "sex": le_sex}, MODELS_DIR / "encoders.pkl")

# metrics.json se guarda al final, a propósito: el API solo empieza a
# construir su registro cuando este archivo existe, así que escribirlo de
# último garantiza que todos los .pkl que referencia ya estén en disco.
best_model = max(metrics, key=lambda name: metrics[name]["accuracy"])
metrics["_best_model"] = best_model
atomic_json_dump(metrics, MODELS_DIR / "metrics.json")

print(f"\nMejor modelo: {best_model} (accuracy={metrics[best_model]['accuracy']})")
print(f"Artefactos guardados en {MODELS_DIR}")


Mejor modelo: random_forest (accuracy=1.0)
Artefactos guardados en /models
